# Preliminary data exploration

The file \src\data\extract_cities.py analyses the raw dataset by loading the original csv files into temporary DuckDB tables. These are then filtered for specific cities (defined inside extract_cities.py) and the corresponding compressed parquet files are built.

In order to inspect what the files contain let's first define the working path.

In [1]:
from pathlib import Path

cwd = Path.cwd()
for folder in (cwd, *cwd.parents):
    if (folder / "src" / "data" / "extract_cities.py").is_file():
        PROJECT_ROOT = folder
        break
else:
    raise RuntimeError("Start Jupyter from the CityFlow folder or one of its subfolders.")

For each city we have the following files:

1) **cities** is single-row df containing : city id, name, coordinates (latitude and longitude), timezone, country and return_to_official_only (True = bikes can only be returned to official dock stations, False = bikes can be returned everywhere in designated zones)
2) **station_status** is a df containing : station_id, time, total numb of bikes, booked bikes, bikes available to rent, free racks, free special racks, maintenance status
3) **stations** is a df containing : id (equivalent to station_id), city_id (identical for each row), name, app_number, **terminal_type (not sure what this represents**), place_type, # of bike_racks, # of special_racks, rack_locks and coordinates (latitude and longitude)
4) **trips** is a df containing : bike_id, city_id (identical for each row), time_start, coordinates_start (latitude and longitude), coordinates_end (latitude and longitude), station_id_start, station_id_end, battery_start, battery_end, duration (seconds), distance (meters)

In [3]:
import duckdb

path = PROJECT_ROOT / "data" / "karlsruhe" / "trips.parquet"
duckdb.read_parquet(str(path)).limit(5).df()

,bike_id,city_id,time_start,lon_start,lat_start,lon_end,lat_end,station_id_start,station_id_end,battery_start,battery_end,duration,distance
0,54000,21,1.665046e+09,8.389058,49.002098,8.400361,48.994617,<NA>,264394,None,None,360.0,1171.289441
1,54000,21,1.665047e+09,8.400361,48.994617,8.441567,49.028271,264394,<NA>,None,None,1500.0,4799.545447
2,54000,21,1.665049e+09,8.441537,49.028280,8.405018,49.010687,<NA>,392504,None,None,960.0,3304.346604
3,54000,21,1.665068e+09,8.405018,49.010687,8.403639,49.006022,392504,<NA>,None,None,4920.0,528.386597
4,54000,21,1.665074e+09,8.403602,49.006018,8.386730,48.992382,<NA>,<NA>,None,None,1740.0,1952.947166


**Initially, I would focus on a city where return_to_official_only=True. This way the position of available bikes should be well-defined at any moment.**

**Idea :** we could make a classification model that tells you whether a specific journey is possible in a fixed time (e.g. 2 hours from now). Based on start and end dock station coordinates, we should predict the probability that at least 2 bikes are available in station_start 2 hours from now, and the probability that at least 2 racks are free in station_end 2 hours + travel time from now. The avg travel time between two stations can be computed from trips.parquet.

## Inspecting Data Quality

**Check the return_to_official policy**

For cities where return_to_official_only is True, we would expect that each trip starts and ends in official dock stations. In practice, we find:
- Heidelberg : 12.9% (return_to_official_only = True)
- Bergamo :    12.9% (return_to_official_only = True)
- Karlsruhe :  97.1% (return_to_official_only = False) 

In [8]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))
from src.inspection.trips import count_trips_missing_station_ids

city_dir = PROJECT_ROOT / "data" / "karlsruhe"
trips_path = city_dir / "trips.parquet"

count_trips_missing_station_ids(city_dir) / duckdb.execute('SELECT COUNT(*) FROM read_parquet(?)', [str(trips_path)]).fetchall()[0][0]

0.970818118514241

**Check the bike usage during the year**

1) First method can count how many dock stations changes there are for each day in the spanned period